# Day 11 — ILT 4: Data Governance – Unity Catalog

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Calendar slot** | Day 11 &middot; ILT 4 |
| **Duration** | 60 minutes |
| **Builds on** | Day 6/7 (`gbmart.gold` dimensions + `fact_sales`), Day 9/10 (Delta version history via `DESCRIBE HISTORY`) |
| **Reads (real, read-only)** | Grants on the real `gbmart` catalog/schema/table, real version history of `gbmart.gold.dim_customer` |
| **Writes (practice only)** | One synthetic table + two functions in your own `main.YOUR_SCHEMA` &mdash; **never** `gbmart.*` |
| **Deferred hands-on** | *Apply Unity Catalog Governance — GRANT/REVOKE, Mask PII, Verify Lineage (4 Sources)* — a separate, later session that applies today's concepts to the real Gold layer. Not built today. |

### Learning Objectives
- Explain Unity Catalog's permission hierarchy (metastore &rarr; catalog &rarr; schema &rarr; table/view/function) and why grants are inherited downward
- Write `GRANT`/`REVOKE` statements for users, groups, and service principals using the standard privilege set
- Read real grants and real version history off the live `gbmart` catalog — safely, with nothing but `SHOW GRANTS`/`DESCRIBE HISTORY`
- Build a real row filter and a real column mask, end to end, on a practice table — and explain why that practice table exists instead of touching `gbmart` directly
- State the precise difference between `DESCRIBE HISTORY` (one table's own version timeline) and Unity Catalog lineage (the cross-table/notebook/job graph in Catalog Explorer)

---
**A note on safety, upfront:** every `ALTER TABLE`, `CREATE FUNCTION`, `GRANT`, and `REVOKE` that actually *runs* in this notebook targets only a practice schema you create yourself (`main.YOUR_SCHEMA`) — never any `gbmart.*` object. The only things this notebook does against the real `gbmart` catalog are `SHOW GRANTS`, `DESCRIBE HISTORY`, and plain `SELECT`s — all non-destructive, metadata-only or read-only. Nothing here starts a job, workflow, pipeline, SQL warehouse, or cluster.

**Instructions:** Run each cell in order with **Shift + Enter**, or **Run All**. No blanks to fill in today — every cell runs as-is.

## Why This, Now

By Day 7 you had one shared `gbmart.gold` layer built from GlobalMart's 2 real ingestion pathways (ADLS Autoloader files and the Postgres/Supabase CDC pipeline — Day 3's REST API/GraphDB exploration never feeds this layer, see Day 3 ILT 1 / Day 4 ILT 1). That's exactly the moment governance stops being theoretical: the same `fact_sales` and `dim_customer` tables now get queried by data engineers, analysts, a future Genie space (Day 12's topic), and eventually external auditors — and not all of them should see the same thing. An analyst needs `SELECT` on `gold`, not `MODIFY`. A regional sales lead should only see their own region's rows. Nobody outside HR should see a raw customer email address. Unity Catalog gives you exactly four mechanisms for this, and this session covers all four: **permissions** (who can touch what), **row filters** (which rows they see), **masking** (which column values they see), and **lineage** (proving, after the fact, where a number actually came from).

## 1. The Permissions Model — One Hierarchy, Four Levels

Unity Catalog secures exactly one hierarchy, top to bottom. Every privilege you grant sits at one of these four levels:

```
metastore                                    (one per region/account -- you don't manage this)
  |-- gbmart                                 CATALOG
        |-- bronze                           SCHEMA
        |     |-- customers, orders, ...     TABLE
        |-- silver                           SCHEMA
        |     |-- customers, orders, ...     TABLE
        |-- gold                             SCHEMA
              |-- dim_customer                TABLE
              |-- dim_product                 TABLE
              |-- fact_sales                  TABLE
              |-- ...
```

Functions (including the row-filter and masking functions later in this notebook) and views sit at the same level as tables — they're securable objects inside a schema too.

### Principals and the Standard Privilege Set

Unity Catalog grants to three kinds of **principal** — the syntax is identical no matter which one you're granting to:

| Principal type | Example | Typical use |
|---|---|---|
| User | an individual's login (e.g. an email address) | One person's own access |
| Group | `` `analysts` ``, `` `data_engineers` `` | The normal case — grant to a group, add/remove members, never re-grant |
| Service principal | An application/automation identity (no human login) | Jobs, pipelines, and other automated processes |

| Privilege | Grants the ability to... |
|---|---|
| `USE CATALOG` | "Step into" a catalog at all — a prerequisite for anything below it, grants no access by itself |
| `USE SCHEMA` | Step into a schema — same idea, one level down |
| `SELECT` | Read rows from a table or view |
| `MODIFY` | `INSERT`/`UPDATE`/`DELETE`/`MERGE` rows |
| `CREATE TABLE` | Create new tables inside a schema |
| `CREATE FUNCTION` | Create SQL/Python functions inside a schema — including row-filter and masking functions |
| `CREATE SCHEMA` | Create new schemas inside a catalog |
| `EXECUTE` | Call a function directly |
| `ALL PRIVILEGES` | Every privilege valid at that level, combined |

> `USE CATALOG`/`USE SCHEMA` trip people up: they're **necessary but not sufficient**. Being able to "step into" `gbmart.gold` doesn't let you read anything inside it — you also need `SELECT` on the schema or the individual table.

### `GRANT` / `REVOKE` Syntax, and Inheritance

```sql
-- Give a group read access to an entire schema -- present AND future tables in it
GRANT USE CATALOG ON CATALOG main                    TO `analysts`;
GRANT USE SCHEMA  ON SCHEMA  main.YOUR_SCHEMA         TO `analysts`;
GRANT SELECT      ON SCHEMA  main.YOUR_SCHEMA         TO `analysts`;

-- Narrower: only one specific table, nothing else in the schema
GRANT SELECT ON TABLE main.YOUR_SCHEMA.practice_customers TO `analysts`;

-- Take it away again -- same shape, opposite direction
REVOKE SELECT ON SCHEMA main.YOUR_SCHEMA FROM `analysts`;
```

**Inheritance is the important part.** A privilege granted at the catalog or schema level flows downward automatically:

| Granting `SELECT` at this level... | ...covers |
|---|---|
| `CATALOG gbmart` | Every schema and every table in the whole catalog, now and later |
| `SCHEMA gbmart.gold` | Every table in `gold` today — **and any new dimension or fact table added to `gold` next month, with no new grant needed** |
| `TABLE gbmart.gold.fact_sales` | Only that one table |

That middle row is the one worth memorizing: **grant at the schema level and mean it** — it's not a one-time snapshot of today's tables, it's a standing rule that applies to whatever exists in that schema at query time, including tables nobody has built yet.

> The `GRANT`/`REVOKE` statements above are illustrative syntax only, shown against the practice schema naming pattern — nothing in this notebook actually executes a `GRANT` or `REVOKE` against anything. The only thing we execute against real grants is *reading* them, next.

### Reading the Real `gbmart` Grants — Live, Read-Only

`SHOW GRANTS` is itself just a read — it changes nothing. We'll check three real levels on the real `gbmart` catalog: the catalog itself, the `gold` schema, and one specific table. Each is wrapped in `try/except`: this demo account may not be the owner of `gbmart` (the instructor's account created it), and Unity Catalog only lets you see grants on objects you own or administer — a permission error here is a **normal boundary**, not a bug, and is exactly the kind of thing this session is teaching you to expect.

In [ ]:
CATALOG = "gbmart"

grant_checks = [
    ("CATALOG gbmart",               f"SHOW GRANTS ON CATALOG {CATALOG}"),
    ("SCHEMA gbmart.gold",           f"SHOW GRANTS ON SCHEMA {CATALOG}.gold"),
    ("TABLE gbmart.gold.fact_sales", f"SHOW GRANTS ON TABLE {CATALOG}.gold.fact_sales"),
]

for label, stmt in grant_checks:
    print(f"--- SHOW GRANTS ON {label} ---")
    try:
        spark.sql(stmt).display()
    except Exception as e:
        print(f"Could not read grants on {label} -- this account is probably not the "
              f"owner/admin of that object, which is a normal Unity Catalog boundary, "
              f"not a bug. ({str(e)[:120]})")
    print()

## 2. Row-Level Security — Row Filters

A **row filter** is a SQL function that returns `BOOLEAN` — one row of a table is visible to a given query only if the function returns `true` for that row. You write the function once, attach it to a table with `ALTER TABLE ... SET ROW FILTER`, and from then on **every** query against that table — from any tool, any notebook, any BI dashboard — is filtered automatically. No application code anywhere has to remember to add a `WHERE` clause.

```sql
-- Define: returns true only for rows this filter should let through
CREATE OR REPLACE FUNCTION your_catalog.your_schema.region_filter(region STRING)
RETURNS BOOLEAN
RETURN region = 'US';

-- Attach: from now on, EVERY reader of this table only sees region = 'US' rows
ALTER TABLE your_catalog.your_schema.your_table
SET ROW FILTER your_catalog.your_schema.region_filter ON (region);
```

In a real production deployment the function body is usually role-aware rather than a fixed literal, so different groups see different slices of the *same* table:

```sql
-- Illustrative only -- what a role-aware version looks like in production
CREATE OR REPLACE FUNCTION your_catalog.your_schema.region_filter(region STRING)
RETURNS BOOLEAN
RETURN CASE
    WHEN is_account_group_member('global_admins') THEN true          -- see everything
    WHEN is_account_group_member('us_sales_team')  THEN region = 'US'
    WHEN is_account_group_member('in_sales_team')  THEN region = 'IN'
    ELSE false                                                       -- no group, no rows
END;
```

The hands-on demo later in this notebook uses the simpler fixed-literal version (`region = 'US'`) on purpose — it's deterministic no matter which account runs it, which is exactly what a repeatable classroom demo needs.

## 3. Column-Level Security — Masking

A **masking function** does the column equivalent of a row filter: instead of returning `true`/`false` per row, it returns a *transformed value of the same type* for one column. Attach it with `ALTER TABLE ... ALTER COLUMN ... SET MASK`, and every reader of that column gets the function's output instead of the raw value — same mechanism, applied per-column instead of per-row.

```sql
-- Define: returns a transformed value, same type as the column (STRING here)
CREATE OR REPLACE FUNCTION your_catalog.your_schema.email_mask(email STRING)
RETURNS STRING
RETURN CONCAT('***', SUBSTRING(email, INSTR(email, '@'), LENGTH(email)));

-- Attach to one column
ALTER TABLE your_catalog.your_schema.your_table
ALTER COLUMN email SET MASK your_catalog.your_schema.email_mask;
```

And, again, the realistic production shape is role-aware — full value for one role, redacted for everyone else, on the exact same column of the exact same table:

```sql
-- Illustrative only -- full email for one role, masked for everyone else
CREATE OR REPLACE FUNCTION your_catalog.your_schema.email_mask(email STRING)
RETURNS STRING
RETURN CASE
    WHEN is_account_group_member('customer_support') THEN email                        -- full value
    ELSE CONCAT('***', SUBSTRING(email, INSTR(email, '@'), LENGTH(email)))              -- redacted
END;
```

The hands-on demo below runs the simpler unconditional version — masked for absolutely everyone, including you — for the same reason as the row filter: it's deterministic regardless of which groups your demo account happens to belong to. Swapping in the `CASE`/`is_account_group_member(...)` shape above is the only change needed to make it role-aware in production.

### Why a Practice Table, Not `gbmart`

Everything above is real, runnable SQL — but every function we actually create and every table alteration we actually run below targets a table **we create ourselves**, in `main.YOUR_SCHEMA` — not anything already sitting in the shared Gold layer.

Two reasons, both concrete:
1. **It's disruptive to everyone else, immediately.** The instant a row filter or mask is attached to a shared table, *every* other student's and the instructor's next query against it is filtered/masked too — mid-class, with no warning.
2. **It isn't a quiet, reversible change.** Unlike a `SELECT`, attaching a row filter or a mask is a standing change to the table's own metadata. Undoing it needs another explicit, deliberate statement to drop it again — it doesn't expire, and it doesn't clean up after itself.

Practicing on your own schema first — exactly what the rest of this notebook does — means you get the real mechanics with zero blast radius, and the *actual* Gold-layer version of this exercise (deferred to a later hands-on) gets done once, deliberately, by someone who means it.

> **Compute note:** row filters and column masks are enforced by Unity Catalog on supported compute only — a SQL warehouse, a cluster in shared access mode, or a sufficiently recent single-user runtime. If your attached cluster doesn't support them, the `ALTER`/`SELECT` cells below print a clear fallback message instead of failing — that's an expected compute limitation, not a bug in this demo.

In [ ]:
# --- Practice schema setup -- isolates this demo from the real gbmart catalog --
# Replace YOUR_SCHEMA with something unique to you if you run this again later
# (e.g. main.governance_lab) -- main.YOUR_SCHEMA as-is is fine for one live demo.
YOUR_SCHEMA = "main.YOUR_SCHEMA"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {YOUR_SCHEMA}")

PRACTICE_TABLE  = f"{YOUR_SCHEMA}.practice_customers"
ROW_FILTER_FUNC = f"{YOUR_SCHEMA}.us_region_filter"
MASK_FUNC       = f"{YOUR_SCHEMA}.email_mask"

print(f"Practice schema : {YOUR_SCHEMA}")
print(f"Practice table  : {PRACTICE_TABLE}")
print(f"Functions       : {ROW_FILTER_FUNC}, {MASK_FUNC}")

### Clean Slate First

This demo is meant to be run more than once (by you, or by every cohort you teach). If a previous run left the row filter, the mask, or the functions attached, recreating them can fail with a dependency error before you ever get the chance to replace them. The cell below tears everything down first and silently ignores "doesn't exist yet" errors — expected, and harmless, on the very first run.

In [ ]:
# --- Clean slate -- makes this demo safe to Run All more than once ---------
# Detach filter/mask before dropping the functions they reference, then drop
# the table and the functions. Every statement is wrapped individually: on the
# very first run, all of these fail because nothing exists yet -- that's fine.
for stmt in [
    f"ALTER TABLE {PRACTICE_TABLE} DROP ROW FILTER",
    f"ALTER TABLE {PRACTICE_TABLE} ALTER COLUMN email DROP MASK",
    f"DROP TABLE IF EXISTS {PRACTICE_TABLE}",
    f"DROP FUNCTION IF EXISTS {ROW_FILTER_FUNC}",
    f"DROP FUNCTION IF EXISTS {MASK_FUNC}",
]:
    try:
        spark.sql(stmt)
    except Exception:
        pass  # expected on the first run -- nothing exists yet to tear down

print("Practice schema clean-slated -- safe to (re)build below.")

### Build the Practice Table

Six rows, three regions, one `email` column — just enough to see both a row filter and a mask do something visible.

In [ ]:
spark.sql(f"""
    CREATE TABLE {PRACTICE_TABLE} (
        customer_id   INT,
        customer_name STRING,
        region        STRING,
        email         STRING
    ) USING DELTA
""")

spark.sql(f"""
    INSERT INTO {PRACTICE_TABLE} VALUES
        (1, 'Asha Rao',   'US', 'asha.rao@example.com'),
        (2, 'Liam Chen',  'US', 'liam.chen@example.com'),
        (3, 'Priya Nair', 'IN', 'priya.nair@example.com'),
        (4, 'Tom Becker', 'UK', 'tom.becker@example.com'),
        (5, 'Wei Zhang',  'IN', 'wei.zhang@example.com'),
        (6, 'Sara Ahmed', 'US', 'sara.ahmed@example.com')
""")

print("Baseline -- no row filter, no mask attached yet -- everyone sees every row, full emails:")
spark.table(PRACTICE_TABLE).orderBy("customer_id").display()

### Step 1 — Attach the Row Filter

`region = 'US'` only, exactly as the concept section described.

In [ ]:
# --- Row filter: only region = 'US' rows become visible from here on -------
try:
    spark.sql(f"""
        CREATE OR REPLACE FUNCTION {ROW_FILTER_FUNC}(region STRING)
        RETURNS BOOLEAN
        RETURN region = 'US'
    """)
    spark.sql(f"ALTER TABLE {PRACTICE_TABLE} SET ROW FILTER {ROW_FILTER_FUNC} ON (region)")
    print(f"Row filter attached: {ROW_FILTER_FUNC} (region = 'US')")
except Exception as e:
    print("Could not attach the row filter -- row filters need a SQL warehouse, a "
          "shared-access-mode cluster, or a recent single-user runtime. Check your "
          f"attached compute's access mode. ({str(e)[:120]})")

In [ ]:
try:
    print("Same table, same SELECT as the baseline -- only region='US' rows come back now:")
    spark.table(PRACTICE_TABLE).orderBy("customer_id").display()
except Exception as e:
    print(f"Could not query the filtered table ({str(e)[:120]}) -- see the compute "
          f"note above about row filter support.")

### Step 2 — Attach the Email Mask

Everything before `@` becomes `***`; the domain stays visible.

In [ ]:
# --- Column mask: redact everything before '@' in the email column ---------
try:
    spark.sql(f"""
        CREATE OR REPLACE FUNCTION {MASK_FUNC}(email STRING)
        RETURNS STRING
        RETURN CONCAT('***', SUBSTRING(email, INSTR(email, '@'), LENGTH(email)))
    """)
    spark.sql(f"ALTER TABLE {PRACTICE_TABLE} ALTER COLUMN email SET MASK {MASK_FUNC}")
    print(f"Column mask attached: {MASK_FUNC} (email column)")
except Exception as e:
    print("Could not attach the column mask -- same compute requirement as the row "
          f"filter above. ({str(e)[:120]})")

In [ ]:
try:
    print("Region filter still applied, AND email is now masked too:")
    spark.table(PRACTICE_TABLE).orderBy("customer_id").display()
except Exception as e:
    print(f"Could not query the masked table ({str(e)[:120]}).")

### What Just Happened

Three identical `SELECT * FROM practice_customers` queries, same table, same account, three different results — purely because of metadata attached to the table, with zero application-level filtering code written anywhere:

| Query | Rows returned | `email` column |
|---|---|---|
| Baseline (no filter/mask) | All 6 | Full value |
| After row filter | Only the 3 `region = 'US'` rows | Full value |
| After column mask | Same 3 rows | `***@example.com` style, for all of them |

**Optional cleanup** (not run automatically — left living on purpose, so you can still inspect the row filter/mask/functions in Catalog Explorer after this session):
```sql
ALTER TABLE main.YOUR_SCHEMA.practice_customers DROP ROW FILTER;
ALTER TABLE main.YOUR_SCHEMA.practice_customers ALTER COLUMN email DROP MASK;
DROP TABLE IF EXISTS main.YOUR_SCHEMA.practice_customers;
DROP FUNCTION IF EXISTS main.YOUR_SCHEMA.us_region_filter;
DROP FUNCTION IF EXISTS main.YOUR_SCHEMA.email_mask;
```

## 4. Lineage — Two Different (Related) Things

"Lineage" gets used loosely, but this session means two specific, different things — and the difference is a common trip-up:

| | `DESCRIBE HISTORY` | Unity Catalog Lineage (Catalog Explorer) |
|---|---|---|
| **Scope** | One table's own version timeline | A cross-object graph: tables, notebooks, jobs, dashboards |
| **Answers** | "What changed in this table, and when?" | "What fed this table, and what consumes it downstream?" |
| **Captured by** | The Delta transaction log — every commit | Unity Catalog observing every query that reads one table and writes another |
| **Granularity** | Table + version | Table-level **and** column-level |
| **How you see it** | SQL: `DESCRIBE HISTORY <table>` | Catalog Explorer's **Lineage** tab, or the `system.access.table_lineage`/`column_lineage` system tables |

Both are automatic — neither needs you to manually tag or document anything — but they are not the same feature. `DESCRIBE HISTORY` is the one you can query and see results from right here in a notebook; the cross-table lineage graph is primarily a Catalog Explorer UI experience (with the system-table escape hatch below, if your workspace has it enabled).

### Real Version History — `gbmart.gold.dim_customer`

`dim_customer` is SCD2 (Day 6/10) — it has genuinely interesting history, since Day 10's SCD2 `MERGE` demo created new versions on top of Day 6's initial build. Real, read-only, wrapped in `try/except` for the same permission-boundary reason as the grants check earlier.

In [ ]:
HISTORY_TABLE = f"{CATALOG}.gold.dim_customer"   # SCD2 -- has real version history to show

try:
    print(f"DESCRIBE HISTORY {HISTORY_TABLE} -- most recent versions:")
    (spark.sql(f"DESCRIBE HISTORY {HISTORY_TABLE}")
        .select("version", "timestamp", "operation", "userName")
        .orderBy("version", ascending=False)
        .limit(10)
        .display())
except Exception as e:
    print(f"Could not read history for {HISTORY_TABLE} -- either this account lacks "
          f"SELECT on it, or it doesn't exist yet in this workspace. "
          f"({str(e)[:120]})")

### Bonus (Optional) — Lineage as a Queryable System Table

Some workspaces additionally expose lineage as **queryable metadata** — `system.access.table_lineage` and `system.access.column_lineage` — if an account admin has enabled the system schema. This is 100% read-only either way. Wrapped in `try/except` because whether it's enabled is an account-level setting this demo account may not have; Catalog Explorer's **Lineage** tab always works regardless of this setting.

In [ ]:
try:
    (spark.table("system.access.table_lineage")
        .filter("source_table_full_name LIKE 'gbmart.%' OR target_table_full_name LIKE 'gbmart.%'")
        .limit(10)
        .display())
except Exception as e:
    print("system.access.table_lineage isn't queryable from this account/workspace "
          "(not enabled, or no permission) -- that's expected in many training "
          f"environments. Catalog Explorer's Lineage tab works regardless. ({str(e)[:100]})")

## Key Takeaways
- Unity Catalog has exactly one permission hierarchy — metastore &rarr; catalog &rarr; schema &rarr; table/view/function — and grants at a higher level are inherited by everything below, including objects created later.
- `GRANT`/`REVOKE` syntax doesn't change based on who you're granting to — users, groups, and service principals all use the same statement shape.
- A row filter is a SQL function returning `BOOLEAN`, attached with `ALTER TABLE ... SET ROW FILTER`; a mask is a SQL function returning a transformed value, attached with `ALTER TABLE ... ALTER COLUMN ... SET MASK`. Both apply to every reader automatically — no application-side filtering code.
- `DESCRIBE HISTORY` (one table's own version timeline) and Catalog Explorer lineage (the cross-table/notebook/job graph) are both automatic, and both real, but they answer different questions.
- Never attach a row filter or a mask, and never change a grant, on a real shared table you don't own outright — build it on your own practice schema first, exactly like this session just did.

## Self-Check
- [ ] I can name all 4 levels of the Unity Catalog hierarchy, in order.
- [ ] I can write a `GRANT` statement that gives a group `SELECT` on a schema.
- [ ] I can explain why granting `SELECT` on a schema today also covers a table added to that schema next month.
- [ ] I can write a row-filter function and attach it with `ALTER TABLE ... SET ROW FILTER`.
- [ ] I can write a masking function and attach it with `ALTER TABLE ... ALTER COLUMN ... SET MASK`.
- [ ] I can state, in one sentence, the difference between `DESCRIBE HISTORY` and Catalog Explorer lineage.
- [ ] I can explain, out loud, why today's row-filter/mask demo ran on a practice table instead of a real Gold-layer table.

## What's Next

**Deferred to a later, separate hands-on session:** *Apply Unity Catalog Governance — GRANT/REVOKE, Mask PII, Verify Lineage (4 Sources)* — the exact techniques from this session, applied for real to the shared `gbmart.gold` layer: granting/revoking real access on real schemas and tables, masking real PII columns (`email`, `phone_number`) on `dim_customer`, attaching a real row filter, and verifying real lineage in Catalog Explorer for a table built from GlobalMart's 2 real ingestion pathways (that calendar title's "4 Sources" is the original spec name — see the "2, not 4" correction in Day 3 ILT 1 / Day 4 ILT 1 / Day 10 ILT 2). That session is deliberately not today — today built the concept and the muscle memory safely, on a table that belongs to you alone.

**Right after this, still on Day 11:** the Hands-On lab — Orchestrate the Pipeline — followed by the closing Data Detectives activity.

---
This completes Day 11 ILT 4. Pairs with `Day11_4_ILT4_Data_Governance_Unity_Catalog.html` for the concept walkthrough shown on screen before this notebook is run live.